# MaskGuard AI - Face Mask Detection System
**By Amina Asghar | Computer Vision Portfolio Project**

---

## Setup Guide

### Step 1 - Runtime Configuration
Go to **Runtime -> Change runtime type -> T4 GPU** before running any cell.

### Step 2 - Dataset Setup
1. Visit https://www.kaggle.com/datasets/andrewmvd/face-mask-detection
2. Upload your `kaggle.json` API token when prompted in the dataset cell
3. The dataset is ~150MB

### Step 3 - Run All Cells in Order
Use **Runtime -> Run all** or execute cells sequentially.

### Step 4 - Launch the Gradio Interface
The final cell launches the interactive UI with a public shareable link.

### Features
- **Real-Time Webcam Streaming** - Live continuous detection, not single-frame capture
- **Multi-Cascade Face Detection** - 4 Haar/LBP cascades + CLAHE preprocessing for maximum face detection
- **Tunable Sensitivity** - Adjust detection sensitivity from the UI (no code changes needed)
- **Optimized Pipeline** - Faster inference for smoother streaming

### Important Notes
- Dataset contains ~853 annotated images in PASCAL VOC XML format
- Labels: `with_mask`, `without_mask`, `mask_weared_incorrect`
- HOG+SVM baseline runs on CPU; MobileNetV2 fine-tuning requires GPU (T4)
- Webcam streaming in Gradio requires browser camera permissions
- If training loss stays high after epoch 5, verify the dataset path in the config cell

---

In [ ]:
!pip install "gradio>=4.44.0" opencv-python-headless torch torchvision pillow numpy matplotlib scikit-learn scikit-image tqdm kaggle lxml -q

In [ ]:
import os
import glob
import random
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import gradio as gr
import warnings
import time
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU mode"}')
print(f'Gradio version: {gr.__version__}')

In [ ]:
from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle API configured.')

In [ ]:
!kaggle datasets download -d andrewmvd/face-mask-detection --unzip -p /content/facemask
print('Dataset downloaded.')
!ls /content/facemask

In [ ]:
IMAGES_DIR = '/content/facemask/images'
ANNOTS_DIR = '/content/facemask/annotations'

LABEL_MAP = {
    'with_mask': 0,
    'without_mask': 1,
    'mask_weared_incorrect': 2
}
LABEL_NAMES = ['With Mask', 'Without Mask', 'Incorrect Mask']
LABEL_COLORS = [(0, 220, 100), (0, 60, 255), (0, 165, 255)]

def parse_annotation(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bb = obj.find('bndbox')
        xmin = int(float(bb.find('xmin').text))
        ymin = int(float(bb.find('ymin').text))
        xmax = int(float(bb.find('xmax').text))
        ymax = int(float(bb.find('ymax').text))
        objects.append({'label': name, 'bbox': (xmin, ymin, xmax, ymax)})
    return filename, objects

all_annotations = {}
xml_files = glob.glob(os.path.join(ANNOTS_DIR, '*.xml'))

for xml_path in xml_files:
    filename, objects = parse_annotation(xml_path)
    all_annotations[filename] = objects

print(f'Annotations loaded: {len(all_annotations)} images')

label_counts = {'with_mask': 0, 'without_mask': 0, 'mask_weared_incorrect': 0}
for objs in all_annotations.values():
    for obj in objs:
        if obj['label'] in label_counts:
            label_counts[obj['label']] += 1

for k, v in label_counts.items():
    print(f'  {k}: {v}')

In [ ]:
def extract_face_patches(images_dir, annotations, target_size=(64, 64)):
    patches, labels, raw_labels = [], [], []
    for filename, objects in tqdm(annotations.items(), desc='Extracting patches'):
        img_path = os.path.join(images_dir, filename)
        if not os.path.exists(img_path):
            continue
        img = cv2.imread(img_path)
        if img is None:
            continue
        for obj in objects:
            label = obj['label']
            if label not in LABEL_MAP:
                continue
            xmin, ymin, xmax, ymax = obj['bbox']
            xmin, ymin = max(0, xmin), max(0, ymin)
            xmax = min(img.shape[1], xmax)
            ymax = min(img.shape[0], ymax)
            if xmax <= xmin or ymax <= ymin:
                continue
            patch = img[ymin:ymax, xmin:xmax]
            if patch.size == 0:
                continue
            patch_resized = cv2.resize(patch, target_size)
            patches.append(patch_resized)
            labels.append(LABEL_MAP[label])
            raw_labels.append(label)
    return patches, labels, raw_labels

all_patches, all_labels, all_raw = extract_face_patches(IMAGES_DIR, all_annotations)
print(f'Total face patches: {len(all_patches)}')